# Object-Oriented Programming in Python

## Learning objectives
By the end of the lesson, you should be able to:
- Distinguish a class from an instance.
- Create objects with an initial state defined in `__init__`.
- Explain the role of `self` and use instance methods.
- Distinguish class attributes from instance attributes.
- Identify shared lists and create independent state when necessary.
- Choose between instance, class, and static methods.
- Implement useful representations with `__str__` and `__repr__`.





## 1. What is object-oriented programming for?

Object-oriented programming (OOP) lets us organize related data and operations within the same structure.

For example, a bank account can have a number and a balance. Deposit and withdrawal operations use that state and can apply rules, such as preventing a withdrawal greater than the available balance.

| Concept | Meaning | Example |
|---|---|---|
| Class | Defines a type and the behavior of its instances | `BankAccount` |
| Instance or object | A particular object created from a class | `anna_account` |
| Attribute | Stores information associated with an object or a class | `anna_account.balance` |
| Method | An operation provided by the class | `anna_account.deposit(50)` |

OOP does not replace all functions and dictionaries. It is especially useful when we need to represent several entities with similar structure, behavior, and rules.

## 2. A class and an instance are not the same thing

A class defines how its instances are created and used. Two instances of the same class can have different states.

Consider a `Car` class: every car can have a make and a color, but each object stores its own values.

A class is itself an object in Python. Calling it a “blueprint” is therefore a useful analogy, not a claim that the class cannot be used directly.


One class defines the structure; two instances can store different values for the same attributes.

### The simplest class

The `class` keyword begins the definition. `pass` is an empty statement that provides a valid body until we add behavior.

`EmptyCar()` creates an instance. The parentheses are part of the call.

In [ ]:
class EmptyCar:
    pass

empty_car = EmptyCar()
print(type(empty_car).__name__)
print(isinstance(empty_car, EmptyCar))

EmptyCar
True


### Names and dot notation

By convention, class names use `CapWords`, such as `BankAccount`. Method and attribute names normally use `snake_case`, such as `show_balance`.

This convention does not prohibit digits: `Employee2026`, for example, can be a valid identifier. Do not use spaces or begin the name with a digit.

The dot accesses attributes: `object.attribute`. The attributes in this lesson are public. By convention, a leading `_` indicates a non-public part of an interface; it does not itself create an access barrier.

## 3. Initializing objects with `__init__`

The `__init__` method prepares the state of a newly created instance. Arguments supplied when calling the class are used to initialize its attributes.

It is often called a “constructor” in introductory explanations. More precisely, it is the initializer: creating an instance and initializing it are distinct stages. It must not return an object; normally it contains no `return` statement.

In this class, `self.make` and `self.color` are instance attributes.

In [ ]:
class Car:
    def __init__(self, make, color):
        self.make = make
        self.color = color

    def paint(self, new_color):
        self.color = new_color

blue_car = Car("Toyota", "blue")
red_car = Car("Fiat", "red")

print(blue_car.make, blue_car.color)
print(red_car.make, red_car.color)

Toyota blue
Fiat red


### Reading an object creation call

In `Car("Toyota", "blue")`:
1. An instance of `Car` is created.
2. `__init__` receives that instance through its first parameter, named `self` here.
3. The arguments supply the values for `make` and `color`.
4. The assignments store those values in the instance.

`make` is a local parameter. `self.make` is an attribute that remains available on the object after the initializer finishes.

## 4. What does `self` mean?

In a regular instance method, the first parameter receives the instance associated with the call. The name `self` is a Python convention, not a reserved word.

In `blue_car.paint("green")`, Python supplies `blue_car` as the first argument. We do not supply it again explicitly.

In this example, these forms have the same effect:

```python
blue_car.paint("green")
Car.paint(blue_car, "green")
```

Prefer the first form in ordinary code.

In a method call, `self` identifies the instance whose state will be used or changed.

### Predict before running

When we paint `blue_car`, does the color of `red_car` also change?

In [ ]:
blue_car.paint("green")

print("First car:", blue_car.color)
print("Second car:", red_car.color)

First car: green
Second car: red


### Two names can refer to the same object

Creating another variable does not automatically create another object. `is` tests whether two references identify the same object.

In [ ]:
another_name = blue_car
another_name.paint("black")

print(another_name is blue_car)
print(blue_car.color)
print(blue_car is red_car)

True
black
False


## 5. Instance attributes and default values

Data attributes do not need to be declared beforehand in the class body. They are created when values are assigned to them.

For introductory examples, a good practice is to establish in `__init__` the attributes that every instance should have. This makes the initial state explicit.

Optional parameters let us create objects with default values or supply different values.

In [ ]:
class Employee:
    def __init__(self, identifier=None, salary=0, department=None):
        self.identifier = identifier
        self.salary = salary
        self.department = department

employee_without_data = Employee()
ricardo = Employee("3789", 2500, "Human Resources")

print(vars(employee_without_data))
print(vars(ricardo))

{'identifier': None, 'salary': 0, 'department': None}
{'identifier': '3789', 'salary': 2500, 'department': 'Human Resources'}


`vars(object)` shows the attribute dictionaries of these instances. This does not guarantee that every Python object provides such a dictionary.

Changing an attribute of `ricardo` does not change the corresponding attribute of `employee_without_data`, because these values were assigned separately to the instances.

In [ ]:
ricardo.salary = 2700

print("Ricardo’s salary:", ricardo.salary)
print("Other employee’s salary:", employee_without_data.salary)

Ricardo’s salary: 2700
Other employee’s salary: 0


### Caution: a name on its own does not declare an attribute

Writing a bare name in the class body does not create an attribute “without a value.” It attempts to evaluate that name. If the name is undefined, `NameError` occurs while the class body executes. This is not a rule requiring all attributes to be declared beforehand.

Use an assignment, such as `self.salary = salary`, to establish the attribute.

In [ ]:
try:
    class IncompleteExample:
        undefined_attribute
except NameError:
    print("The name is undefined: an assignment is needed.")

The name is undefined: an assignment is needed.


## 6. Methods: operations on state

A method can inspect state, change it, or return a result. Not every method needs to modify attributes.

Names should describe the operation. It helps to distinguish:
- Querying: returning information without changing the object.
- Updating: changing the object's state.
- Validating: rejecting an operation that violates the example's rules.

We will represent a person with a name, age, weight in kilograms, and height in meters.

### A simple model of a person

This exercise uses an artificial growth rule: for each simulated year that ends at an age below 21, height increases by 0.5 cm. This equals `0.005` meters. It is not a biological model.

The loop applies the rule year by year, even when `years` is greater than 1. The other methods receive quantities in the stated units.

In [ ]:
class Person:
    def __init__(self, name, age, weight, height):
        if age < 0 or weight <= 0 or height <= 0:
            raise ValueError("Invalid age, weight, or height.")
        self.name = name
        self.age = age
        self.weight = weight
        self.height = height

    def grow_older(self, years=1):
        if not isinstance(years, int) or years < 0:
            raise ValueError("The number of years must be a non-negative integer.")
        for _ in range(years):
            self.age += 1
            if self.age < 21:
                self.grow(0.005)

    def grow(self, meters):
        if meters < 0:
            raise ValueError("Growth cannot be negative.")
        self.height += meters

    def gain_weight(self, kilograms):
        if kilograms <= 0:
            raise ValueError("The amount must be positive.")
        self.weight += kilograms

    def lose_weight(self, kilograms):
        if kilograms <= 0 or kilograms >= self.weight:
            raise ValueError("The amount must be positive and less than the current weight.")
        self.weight -= kilograms

    @staticmethod
    def calculate_bmi(weight, height):
        if weight <= 0 or height <= 0:
            raise ValueError("Weight and height must be positive.")
        return weight / height ** 2

### Observe the state before and after

Predict the final values. Two years are simulated, from age 19 to 21. Under the defined rule, only the year ending at age 20 produces growth.

The methods that change state have no explicit `return`, so they return `None`. Changing an object and returning a result are different choices.

In [ ]:
person_model = Person("Diogo", 19, 80, 1.80)
print("Before:", vars(person_model))

result = person_model.grow_older(2)
person_model.gain_weight(2)
person_model.lose_weight(1)

print("Final age:", person_model.age)
print(f"Final height: {person_model.height:.3f} m")
print("Final weight:", person_model.weight)
print("Return value of grow_older:", result)

Before: {'name': 'Diogo', 'age': 19, 'weight': 80, 'height': 1.8}
Final age: 21
Final height: 1.805 m
Final weight: 81
Return value of grow_older: None


## 7. Integrated example: a bank account

The following class brings together state and rules:
- An account has a number, branch, balance, and type.
- Its type must be one of the defined codes.
- The initial balance cannot be negative.
- A deposit or withdrawal must have a positive value.
- A withdrawal cannot exceed the balance.

In [ ]:
class BankAccount:
    types = {"01": "Checking account", "02": "Savings account"}

    def __init__(self, number, branch, initial_balance=0, type_code="01"):
        if type_code not in self.types:
            raise ValueError("Invalid account type.")
        if initial_balance < 0:
            raise ValueError("The initial balance cannot be negative.")
        self.number = number
        self.branch = branch
        self.balance = initial_balance
        self.type_code = type_code

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("The deposit must be positive.")
        self.balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("The withdrawal must be positive.")
        if amount > self.balance:
            raise ValueError("Insufficient balance.")
        self.balance -= amount

    def get_balance(self):
        return self.balance

In [ ]:
account = BankAccount("0001-0", "001", initial_balance=100)
print("Initial balance:", account.get_balance())

account.deposit(50)
account.withdraw(30)
print("Final balance:", account.get_balance())

Initial balance: 100
Final balance: 120


### Reject invalid operations

When a withdrawal is refused, the balance should not change. We handle the exceptions so the notebook can continue running.

The values used are small integers. We are not discussing monetary rounding or building an accounting system.

In [ ]:
try:
    account.withdraw(1000)
except ValueError as error:
    print(error)

print("Balance after the attempt:", account.get_balance())

try:
    invalid_account = BankAccount("0002-0", "001", type_code="99")
except ValueError as error:
    print(error)

Insufficient balance.
Balance after the attempt: 120
Invalid account type.


### Encapsulating a rule does not make an attribute private

The methods above organize operations and validation, but `balance` remains a public attribute. Direct assignment could bypass those rules.

Using a class does not therefore automatically prevent every external change. In later lessons, properties using `@property` can help control certain operations. In this lesson, we distinguish a data attribute from that specific Python feature.

## 8. Class attributes and instance attributes

An attribute defined in the class body can be shared by instances. An attribute assigned to `self` can store state belonging to a particular instance.

In the following example:
- `team_name` belongs to the class.
- `name` belongs to each instance.

For these simple data attributes, when an instance has no attribute of its own with that name, the lookup continues in the class.

In [ ]:
class Player:
    team_name = "Liverpool"

    def __init__(self, name):
        self.name = name

p1 = Player("Mark")
p2 = Player("Steve")

print(p1.name, p1.team_name)
print(p2.name, p2.team_name)
print(Player.team_name)

Mark Liverpool
Steve Liverpool
Liverpool


### Change the class or assign to an instance?

`Player.team_name = ...` changes the class attribute.

In this example, `p1.team_name = ...` creates an attribute on `p1`. It does not change `Player.team_name`; the new value on `p1` takes priority when accessed through that instance.

Predict what happens when the class changes again after `p1` has its own value.

In [ ]:
Player.team_name = "Manchester City"
print("After changing the class:", p1.team_name, p2.team_name)

p1.team_name = "Porto"
print("After assigning to p1:", p1.team_name, p2.team_name)

Player.team_name = "Liverpool"
print("After changing the class again:", p1.team_name, p2.team_name)
print("Attributes of p1:", vars(p1))

After changing the class: Manchester City Manchester City
After assigning to p1: Porto Manchester City
After changing the class again: Porto Liverpool
Attributes of p1: {'name': 'Mark', 'team_name': 'Porto'}


## 9. The risk of shared lists

A list created in the class body is a single object. If several instances use it without creating their own lists, they all observe mutations to that list.

This is not necessarily an error. It is an error when the intention was to store a different history for each player.

In [ ]:
class PlayerWithSharedHistory:
    former_teams = []

    def __init__(self, name):
        self.name = name

shared_1 = PlayerWithSharedHistory("Mark")
shared_2 = PlayerWithSharedHistory("Steve")
shared_1.former_teams.append("Barcelona")
shared_2.former_teams.append("Chelsea")

print(shared_1.former_teams)
print(shared_2.former_teams)
print(shared_1.former_teams is shared_2.former_teams)

['Barcelona', 'Chelsea']
['Barcelona', 'Chelsea']
True


A class-level list is shared; creating a new list in each initializer allows independent histories.

### Create one list per instance

The body of `__init__` runs for each new instance. Therefore, `self.former_teams = []` creates a new list for that object.

The important change is not whether a list is mutable: it is which object each instance refers to.

In [ ]:
class PlayerWithIndividualHistory:
    team_name = "Liverpool"

    def __init__(self, name):
        self.name = name
        self.former_teams = []

individual_1 = PlayerWithIndividualHistory("Mark")
individual_2 = PlayerWithIndividualHistory("Steve")
individual_1.former_teams.append("Barcelona")
individual_2.former_teams.append("Chelsea")

print(individual_1.former_teams)
print(individual_2.former_teams)
print(individual_1.former_teams is individual_2.former_teams)

['Barcelona']
['Chelsea']
False


### Intentional sharing

A class-level list can be appropriate when we want a common record. Here, we store the names added when objects are created.

This record tracks creations in the example; it does not automatically track removals, later name changes, or object lifetimes.

In [ ]:
class PlayerWithRegistry:
    team_members = []

    def __init__(self, name):
        self.name = name
        PlayerWithRegistry.team_members.append(name)

registered_1 = PlayerWithRegistry("Mark")
registered_2 = PlayerWithRegistry("Steve")
print(PlayerWithRegistry.team_members)

['Mark', 'Steve']


## 10. Instance, class, and static methods

The three types differ in the argument Python supplies automatically.

| Type | Declaration | First automatic argument | Common use |
|---|---|---|---|
| Instance | `def method(self, ...)` | The instance | Inspect or change the object |
| Class | `@classmethod` | The class, conventionally `cls` | Create instances or operate on the class |
| Static | `@staticmethod` | None | A calculation related to the domain |

A class method can also be called through an instance. A static method can be called through the class or an instance without that instance being added to its arguments.

Choose a method type according to whether it needs the instance, the class, or only the explicitly supplied arguments.

### A class method as an alternative way to create objects

Sometimes we receive data in a form different from the one expected by `__init__`. A class method can convert the data and create an instance using `cls(...)`.

The following method calculates an approximate age from a birth year and an explicit reference year. It does not account for the birthday's day or month. The reference year is example input, not an automatic lookup of the current date.

In [ ]:
class Student:
    school_name = "ABC School"

    def __init__(self, name, age):
        self.name = name
        self.age = age

    @classmethod
    def from_birth_year(cls, name, birth_year, reference_year):
        if birth_year > reference_year:
            raise ValueError("The birth year cannot be later than the reference year.")
        return cls(name, reference_year - birth_year)

    @classmethod
    def change_school(cls, school_name):
        cls.school_name = school_name

    def describe(self):
        return f"{self.name}: {self.age} years; {self.school_name}"

In [ ]:
jessa = Student("Jessa", 20)
joy = Student.from_birth_year("Joy", 2006, 2026)

print(jessa.describe())
print(joy.describe())
print(isinstance(joy, Student))

Student.change_school("XYZ School")
print(jessa.describe())
print(joy.describe())
print(isinstance(joy, Student))

Jessa: 20 years; ABC School
Joy: 20 years; ABC School
True
Jessa: 20 years; XYZ School
Joy: 20 years; XYZ School
True


A class method receives `cls`, not a particular instance. This does not prevent it from creating objects or using objects supplied explicitly: `from_birth_year()` creates an instance, and its initializer establishes the instance attributes.

### A static method for an independent calculation

`Person.calculate_bmi()` was defined using `@staticmethod`. It receives weight and height but does not automatically receive `self` or `cls`.



A static method is not prohibited from using objects or class names. The difference is that it receives no instance or class implicitly.

In [ ]:
bmi = Person.calculate_bmi(80, 2)
print(f"Calculation result: {bmi:.2f}")

try:
    Person.calculate_bmi(80, 0)
except ValueError as error:
    print(error)

Calculation result: 20.00
Weight and height must be positive.


## 11. Representing objects with `__str__` and `__repr__`

Special methods have names defined by Python's protocols, such as `__init__`, `__str__`, and `__repr__`. Not every name containing underscores is a special method.

| Method | Purpose | Usual call |
|---|---|---|
| `__str__` | A readable presentation for the user | `str(object)` or `print(object)` |
| `__repr__` | An informative representation for inspection and debugging | `repr(object)` or inspection inside collections |

Both must return a string. When practical, `repr()` can produce an expression describing how to reconstruct the object, but this is not a universal guarantee or a requirement for every class.

In the following class, `!r` uses each attribute's representation and includes the quotes needed for string values.

In [ ]:
class PersonRecord:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __str__(self):
        return f"{self.name}, {self.age} years old"

    def __repr__(self):
        return f"PersonRecord({self.name!r}, {self.age!r})"

person = PersonRecord("Joana", 23)
print(person)
print(repr(person))
print([person])

Joana, 23 years old
PersonRecord('Joana', 23)
[PersonRecord('Joana', 23)]


The first result is intended for reading. The second shows the class and relevant values. A list uses its elements' representations, so it displays `PersonRecord(...)`.

In ordinary use, call `str(object)` or `repr(object)` instead of calling the underscore methods directly. If a class does not define `__str__`, the implementation inherited from `object` can use `__repr__`.